# L21 · Decisiones estructurales bajo incertidumbre

**S21 · Lectura: Cap. 25 · 40–55 min**

## La pregunta

¿Qué decisión cambia cuando cambia el intervalo?

## Qué harás

Pasar de un efecto a una decisión con utilidad y coste.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un problema de decisión y un análisis de sensibilidad.

> **Idea clave:** La incertidumbre no es un pie de página: puede cambiar la acción.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, itertools
import matplotlib.pyplot as plt
rng = setup()

In [ ]:
# De un efecto incierto a una decisión
import matplotlib.pyplot as plt
labels = ['efecto', 'incertidumbre', 'utilidad', 'decisión']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("De un efecto incierto a una decisión")
plt.tight_layout(); plt.show()

## 1. El Listado 25.1
Tres políticas candidatas, cada una con un efecto estimado (media, error estándar) y un costo. Utilidad $U=V\cdot\text{efecto}-\text{costo}$, con $V$ = `valor_por_punto_efecto` (cuánto vale, en dinero, un punto de mejora), y $-\infty$ si el costo excede el presupuesto.

In [ ]:
POLITICAS = {
    "sin_intervencion": {"efecto": 0.0, "se": 0.0, "costo": 0},
    "transferencia_simple": {"efecto": 0.12, "se": 0.03, "costo": 500},
    "transferencia_mas_capacitacion": {"efecto": 0.19, "se": 0.05, "costo": 900},
}
def U(efecto, costo, V=3000, presupuesto=1000):
    return -np.inf if costo > presupuesto else V * efecto - costo
def reporte(politicas, V):
    filas = {}
    for k, d in politicas.items():
        filas[k] = {"U": U(d["efecto"], d["costo"], V), "U (IC95% bajo)": U(d["efecto"] - 1.96 * d["se"], d["costo"], V), "U (IC95% alto)": U(d["efecto"] + 1.96 * d["se"], d["costo"], V)}
    return pd.DataFrame(filas).T
rep = reporte(POLITICAS, 3000); show(rep.round(1))
print("acción óptima (utilidad esperada), V = 3000:", rep["U"].idxmax())

**Mira el resultado.** Con V = 3000 todas las acciones sustantivas tienen utilidad negativa. ¿Qué dice la «acción óptima» y por qué es una conclusión frágil?

**Respuesta:**

Dice «sin intervención» porque ninguna acción compensa su costo con ese valor por punto. Es frágil porque $V$ no sale de ningún dato: con otro valor razonable la recomendación cambia (sección siguiente). Reportar la recomendación sin el valor que la sostiene es incompleto.



## 2. Barrido del parámetro normativo (E13, ejercicio 25.6.2.1)
¿Para qué valores de $V$ cambia la acción óptima? Cada acción tiene utilidad lineal en $V$: $U_d(V)=V\,\tau_d-c_d$. La óptima cambia donde se cruzan las rectas:
- sin intervención → transferencia simple cuando $V\cdot0{,}12=500\Rightarrow V\approx4167$;
- transferencia simple → capacitación cuando $V\cdot0{,}07=400\Rightarrow V\approx5714$.

In [ ]:
Vs = np.arange(1000, 9001, 250)
optima = {int(V): reporte(POLITICAS, V)["U"].idxmax() for V in Vs}
cambios = [(V, optima[V]) for i, V in enumerate(sorted(optima)) if i == 0 or optima[V] != optima[sorted(optima)[i - 1]]]
print("cambios de acción óptima (primer V de la malla donde aparece):", cambios)
# umbrales exactos
t1 = 500 / 0.12; t2 = (900 - 500) / (0.19 - 0.12)
print(f"umbrales exactos: sin→simple en V = {t1:.0f} · simple→capacitación en V = {t2:.0f}")
for k, d in POLITICAS.items(): plt.plot(Vs, Vs * d["efecto"] - d["costo"], label=k)
plt.axvline(t1, c="gray", ls=":"); plt.axvline(t2, c="gray", ls=":"); plt.xlabel("V (valor por punto de efecto)"); plt.ylabel("utilidad esperada"); plt.legend(); plt.show()

In [ ]:
# Ejercicio: ¿y si el presupuesto fuera 800? La capacitación (900) deja de ser factible.
rep_b = {V: reporte(POLITICAS, V)["U"] for V in (3000, 5000, 8000)}
presupuesto_800 = {V: max(POLITICAS, key=lambda k: U(POLITICAS[k]["efecto"], POLITICAS[k]["costo"], V, presupuesto=800)) for V in (3000, 5000, 8000)}
print("acción óptima con presupuesto 800:", presupuesto_800)

## 3. Maximin y minimax regret sobre el intervalo (§25.2.3)
Tomamos como **conjunto identificado** el intervalo $\hat\tau_d\pm1{,}96\,\mathrm{SE}_d$ de cada acción y evaluamos en los **vértices** de la caja (es suficiente: la utilidad es lineal en cada efecto y el arrepentimiento es convexo, así que el máximo está en un vértice).

- **Maximin (Wald):** elige la acción con mejor **peor** utilidad: $\max_d\min_s U(d,s)$.
- **Minimax regret (Savage):** el *arrepentimiento* de $d$ en el escenario $s$ es $\max_{d'}U(d',s)-U(d,s)$; elige la acción con el menor **arrepentimiento máximo**.

### Completa tú
`maximin(politicas, V, escenarios)` y `minimax_regret(politicas, V, escenarios)` devuelven el **nombre** de la acción elegida. `escenarios` es una lista de diccionarios `{acción: efecto}`.

In [ ]:
def escenarios_vertices(politicas, z=1.96):
    ext = {k: ([d["efecto"] - z * d["se"], d["efecto"] + z * d["se"]] if d["se"] > 0 else [d["efecto"]]) for k, d in politicas.items()}
    ks = list(ext)
    return [dict(zip(ks, combo)) for combo in itertools.product(*[ext[k] for k in ks])]
ESC = escenarios_vertices(POLITICAS)
print(len(ESC), "escenarios (vértices):", ESC[0])

In [ ]:
# ✏️ Completa tú — maximin y minimax_regret
# Pista: Maximin: para cada acción, el MÍNIMO de U sobre escenarios; elige el máximo de esos mínimos. Regret: mejor U del escenario menos U de la acción; toma el MÁXIMO por acción y elige el menor.
def _util(politicas, V, esc, k, presupuesto=1000):
    return U(esc[k], politicas[k]["costo"], V, presupuesto)

def maximin(politicas, V, escenarios):
    """Nombre de la acción con mejor peor utilidad entre los escenarios."""
    raise NotImplementedError("completa maximin")

def minimax_regret(politicas, V, escenarios):
    """Nombre de la acción con menor arrepentimiento máximo entre los escenarios."""
    raise NotImplementedError("completa minimax_regret")

In [ ]:
lab_check("maximin / minimax regret (casos de juguete)", lambda: (
    maximin({"a": {"efecto": 0, "se": 0, "costo": 0}, "b": {"efecto": .1, "se": .1, "costo": 0}}, 1000, escenarios_vertices({"a": {"efecto": 0, "se": 0, "costo": 0}, "b": {"efecto": .1, "se": .1, "costo": 0}})) == "a",
    minimax_regret({"a": {"efecto": 0, "se": 0, "costo": 0}, "b": {"efecto": .5, "se": .1, "costo": 0}}, 1000, escenarios_vertices({"a": {"efecto": 0, "se": 0, "costo": 0}, "b": {"efecto": .5, "se": .1, "costo": 0}})) == "b"),
    [((), (True, True), 0)])

In [ ]:
def tabla_criterios(politicas, Vs, z=1.96):
    esc = escenarios_vertices(politicas, z); filas = {}
    for V in Vs:
        eu = max(politicas, key=lambda k: U(politicas[k]["efecto"], politicas[k]["costo"], V))
        filas[V] = {"utilidad esperada": eu, "maximin": maximin(politicas, V, esc), "minimax regret": minimax_regret(politicas, V, esc)}
    return pd.DataFrame(filas).T.rename_axis("V")
show(tabla_criterios(POLITICAS, [3000, 4500, 5000, 6000, 8000]))

## 4. Cuando crece la incertidumbre, los criterios divergen (ejercicio 25.6.2.2)
Subimos el error estándar de la capacitación de 0,05 a 0,10 y repetimos.

In [ ]:
POL2 = {k: dict(v) for k, v in POLITICAS.items()}; POL2["transferencia_mas_capacitacion"]["se"] = 0.10
Vs2 = [3000, 4500, 5000, 5800, 6000, 8000]
a, b = tabla_criterios(POLITICAS, Vs2), tabla_criterios(POL2, Vs2)
show(pd.concat({"SE capacitación = 0,05": a, "SE capacitación = 0,10": b}, axis=1))
def n_diverge(se_cap, grilla=range(2000, 15001, 250)):
    pol = {k: dict(v) for k, v in POLITICAS.items()}; pol["transferencia_mas_capacitacion"]["se"] = se_cap
    esc = escenarios_vertices(pol); n = 0
    for V in grilla:
        eu = max(pol, key=lambda k: U(pol[k]["efecto"], pol[k]["costo"], V)); n += len({eu, maximin(pol, V, esc), minimax_regret(pol, V, esc)}) > 1
    return n, len(grilla)
for se_ in (0.03, 0.05, 0.10):
    n_, tot_ = n_diverge(se_); print(f"SE de la capacitación = {se_:.2f}: los tres criterios NO coinciden en {n_} de {tot_} valores de V")

**Mira el resultado.** ¿Cuándo coinciden los tres criterios y qué te dice el patrón sobre cuándo importa elegir uno?

**Respuesta:**

Coinciden cuando una acción domina en casi todo el intervalo (valores de $V$ muy bajos o muy altos). Divergen en la zona de transición, y más cuando crece la incertidumbre: la utilidad esperada se fía del punto estimado; el maximin se vuelve conservador (prefiere no hacer nada si el peor caso es dañino); el minimax regret pondera lo que se pierde por equivocarse. **Cuál criterio se aplicó debe reportarse junto al efecto**.



## 5. La acción que casi siempre falta: «experimentar» (§25.2.3)
Si la incertidumbre es lo que limita, un experimento que **revele** los efectos tiene valor. Estilizamos: pagar `costo_exp` por persona y conocer los efectos verdaderos antes de decidir. Su utilidad es $E[\max_d U_d(\theta)]-\text{costo}_{exp}$, con $\theta\sim N(\hat\tau,\mathrm{SE}^2)$ (independientes).

Es una cota superior del valor de un experimento real (que no elimina toda la incertidumbre).

In [ ]:
def valor_experimentar(politicas, V, costo_exp, n_sim=20000, seed=0):
    r = np.random.default_rng(seed); ks = list(politicas)
    draws = {k: r.normal(politicas[k]["efecto"], politicas[k]["se"], n_sim) if politicas[k]["se"] > 0 else np.full(n_sim, politicas[k]["efecto"]) for k in ks}
    Umat = np.column_stack([np.where(politicas[k]["costo"] > 1000, -np.inf, V * draws[k] - politicas[k]["costo"]) for k in ks])
    con_info = Umat.max(1).mean() - costo_exp
    sin_info = max(U(politicas[k]["efecto"], politicas[k]["costo"], V) for k in ks)
    return con_info, sin_info
filas = {}
for V in (3000, 4500, 5000, 6000, 8000):
    ci, si = valor_experimentar(POL2, V, costo_exp=100); filas[V] = {"U mejor acción (sin información)": si, "U experimentar (costo 100)": ci, "¿conviene experimentar?": "sí" if ci > si else "no"}
show(pd.DataFrame(filas).T.round(1))

**Mira el resultado.** ¿Cuándo vale la pena experimentar y cuándo no? ¿Qué pasa si el experimento cuesta 400 en lugar de 100?

**Respuesta:**

Vale la pena cuando hay **mucha incertidumbre relevante para la decisión**: valores de $V$ cerca de los umbrales (donde equivocarse cuesta más) y errores estándar grandes. Lejos de los umbrales, la mejor acción ya es clara y la información no cambia la decisión. Con un costo de 400 la ventana en que conviene se reduce o desaparece. Omitir «experimentar» del conjunto de acciones sesga la decisión hacia intervenir con la información actual.



## 6. Conexión con el Cap. 21 (ejercicio teórico 25.6.1.1)
El problema de **a quién tratar bajo presupuesto** (L22) es un caso particular: el conjunto de acciones $\mathcal D$ son las asignaciones $\pi:\mathcal X\to\{0,1\}$ con gasto ≤ presupuesto, y la utilidad es lineal, $U(\pi)=\sum_i \pi_i\,\tau_i$ (valor = efecto, sin costo monetario por unidad más allá del presupuesto). Aquí la utilidad se vuelve explícita y no lineal en el parámetro $V$; allí está fijada.

## 7. Supuesto y bitácora

In [ ]:
esc = escenarios_vertices(POL2)
fila_b = bitacora(
    pregunta="¿Qué política de transferencia conviene y cuánto depende la recomendación del valor por punto de efecto?",
    metodo="Utilidad esperada, maximin y minimax regret sobre τ̂ ± 1,96·SE; barrido de V; valor de experimentar",
    estimando="Utilidad esperada de cada política",
    numero_ic=f"con V=3000: «{reporte(POLITICAS, 3000)['U'].idxmax()}»; cambia a transferencia simple en V≈{t1:.0f} y a capacitación en V≈{t2:.0f}",
    supuesto=supuesto("no observable", "declarado y acotado", "el valor por punto de efecto (V) y los costos son supuestos normativos que ningún dato fija; se reporta el barrido"),
    decision="continuar", lab="L21")

---
## Qué queda para el libro
Quedan en el libro: la identificación parcial de Manski con cotas, la utilidad con equidad distributiva (ejercicio teórico 25.6.1.2) y la incertidumbre estructural sobre el SCM. El `DecisionProblem` del lab anterior queda como andamio opcional.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*